# SLAM-ASR: Türkçe Mozilla Common Voice Konuşma Tanıma
### Mozilla Data Collective (MDC) & Modern Türkçe Çok Modlu ASR

> **Referans Makale:** *An Embarrassingly Simple Approach for LLM with Strong ASR Capacity* ([arXiv:2402.08846](https://arxiv.org/abs/2402.08846))
> **Veri Seti:** Mozilla Common Voice (Turkish) via [Mozilla Data Collective](https://mozilladatacollective.com)
> **MDC Dataset ID:** `cmu5wkah500c2o10719lem8m8`

Bu notebook, Google Colab (ücretsiz T4 veya ücretli A100/L4) üzerinde **Mozilla Common Voice Türkçe** veri seti ile SLAM-ASR modelini kurmak, eğitmek ve test etmek için hazırlanmıştır.

---
### Temel Mimarisi:
* **Akustik Kodlayıcı:** `openai/whisper-large-v3` (veya `facebook/mms-300m`) - **Dondurulmuş (Frozen)**
* **Dil Modeli (LLM):** `Qwen/Qwen2.5-7B-Instruct` (veya `Trendyol/Trendyol-LLM-7b-chat-v1.0`) - **Dondurulmuş (Frozen)**
* **Projektör:** $k=5$ Ardışık Çerçeve Birleştirme (50 Hz $\to$ 10 Hz) + Tek Lineer Katman - **Eğitilen Tek Bileşen** (~%0.6 parametre)

## 1. GPU ve Donanım Kontrolü
Colab oturumunuzun GPU hızlandırıcıya bağlı olduğunu doğrulayın (Menü: `Runtime` -> `Change runtime type` -> `T4 GPU` veya `A100 GPU`).

In [ ]:
# GPU kullanılabilirliğini kontrol edin
!nvidia-smi

import torch
print(f"[*] PyTorch Versiyonu: {torch.__version__}")
print(f"[*] CUDA Kullanılabilir mi?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Aktif GPU: {torch.cuda.get_device_name(0)}")
    print(f"[*] VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")


## 2. Depo Dosyalarını Hazırlama ve Bağımlılıkları Yükleme
GitHub reposunu çekin veya çalışma dizinine geçin ve gerekli paketleri yükleyin.

In [ ]:
import os

# Eğer doğrudan GitHub'dan klonlayacaksanız veya çalışma dizininde değilseniz:
if not os.path.exists("slam_asr") and not os.path.exists("requirements.txt"):
    if not os.path.exists("slam-asr-reimplement"):
        !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    if os.path.exists("slam-asr-reimplement/tr/cv"):
        %cd slam-asr-reimplement/tr/cv
    elif os.path.exists("tr/cv"):
        %cd tr/cv
    elif os.path.exists("cv"):
        %cd cv
elif os.path.exists("tr/cv"):
    %cd tr/cv
elif os.path.exists("cv"):
    %cd cv

# Bağımlılıkları yükleyin
!pip install -q -r requirements.txt
!pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate datacollective mutagen


## 3. Kod Dosyalarının Doğrulanması ve Birim Testler
Birim testleri koşturarak modüllerin ve Common Voice veri işleme mantığının hazır olduğunu doğrulayın.

In [ ]:
!python run_tests.py


## 4. Hızlı Doğrulama: Sentetik Türkçe Veri ile Test Eğitimi (30 Saniye)
Büyük veri setini indirmeden önce tüm boru hattının sorunsuz çalıştığını sentetik verimizle test edin.

In [ ]:
# 1. Sentetik Türkçe test verisini üretin
!python scripts/create_toy_turkish_dataset.py

# 2. Hızlı doğrulama eğitimini koşturun (50 adım)
!python train.py --config config/train_toy_debug_turkish.yaml


## 5. Mozilla Common Voice (Türkçe) Veri Seti İndirme ve Hazırlığı
Colab Secrets (Sol paneldeki anahtar simgesi 🔑) üzerinden `MDC_API_KEY` anahtarınız okunur ve Mozilla Data Collective API'si aracılığıyla dataset (`cmu5wkah500c2o10719lem8m8`) otomatik olarak indirilip hazırlanır.

In [ ]:
from google.colab import userdata
import os

# Colab Secrets'tan MDC_API_KEY anahtarını alın
try:
    mdc_api_key = userdata.get('MDC_API_KEY')
    print("[*] MDC_API_KEY anahtari Colab Secrets uzerinden basariyla alindi.")
except Exception:
    mdc_api_key = input("Lutfen Mozilla Data Collective API anahtarinizi (MDC_API_KEY) girin: ")

# Mozilla Common Voice Türkçe veri setini indirin ve hazırlayın
!python scripts/prepare_common_voice.py \
    --download \
    --api_key "{mdc_api_key}" \
    --output_dir data/cv


## 6. Donanıma Göre Otomatik Optimize Olan SLAM-ASR Türkçe Eğitimi
Donanım profilleyicimiz (`device_utils`) GPU'nuza göre en verimli hassasiyeti (bfloat16/float16), kuantizasyonu (4-bit NF4) ve batch boyutunu otomatik olarak belirler.

In [ ]:
# Donanıma göre otomatik optimize edilen Türkçe SLAM-ASR eğitimi
!python train.py --config config/hardware_adaptive_turkish.yaml


## 7. Model Değerlendirmesi (Türkçe WER ve CER Hesaplama)
Eğitilen modelin Common Voice test seti üzerindeki performansını ölçün. Türkçe büyük/küçük harf (`İ -> i`, `I -> ı`) ve noktalama normalizasyonu otomatik uygulanır.

In [ ]:
!python evaluate.py \
    --config config/hardware_adaptive_turkish.yaml \
    --checkpoint outputs/slam_asr_cv_turkish_adaptive/best_checkpoint \
    --manifest data/cv/test.jsonl \
    --output_file eval_results.jsonl \
    --language tr \
    --beam_size 4


## 8. İnteraktif Çıkarım (Test Sesini Dinleme ve Transkribe Etme)
Test setindeki bir Common Voice ses kaydını dinleyin ve modelinizin konuşmayı nasıl yazıya döktüğünü canlı olarak görün.

In [ ]:
import json
import os
import IPython.display as ipd

# Test setinden ilk örneği bulun
sample_audio = None
sample_text = ""

manifest_file = "data/cv/test.jsonl" if os.path.exists("data/cv/test.jsonl") else "data/sample_val_tr.jsonl"
if os.path.exists(manifest_file):
    with open(manifest_file, "r", encoding="utf-8") as f:
        first_line = f.readline()
        if first_line:
            data = json.loads(first_line)
            sample_audio = data.get("audio_filepath")
            sample_text = data.get("text", "")

if sample_audio and os.path.exists(sample_audio):
    print(f"[*] Seçilen Ses: {sample_audio}")
    print(f"[*] Gerçek Metin (Ground Truth): {sample_text}\n")
    display(ipd.Audio(sample_audio))
    
    !python inference.py \
        --audio "{sample_audio}" \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt \
        --prompt "Konuşmayı metne dönüştürün."
else:
    print("[!] Test ses dosyası bulunamadı. Lütfen ses dosyasının yolunu elle belirtin.")


## 9. Zaman Damgalı Google Drive Yedeklemesi
Eğitilen ağırlıkları (`best_checkpoint`), test sonuçlarını (`eval_results.jsonl`), manifestoları ve konfigürasyonu Google Drive'a tarih/saat damgalı olarak kalıcı yedekleyin.

In [ ]:
# 9. Adım: Tüm Model Ağırlıklarını, Manifestoları ve Test Sonuçlarını Drive'a Yedekleme
import os
from datetime import datetime
from google.colab import drive

# 1. Google Drive'ı bağlayın
drive.mount("/content/drive")

# 2. Zaman damgalı hedef klasörü belirleyin
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_target_dir = f"/content/drive/MyDrive/checkpoints/slam_asr_cv_turkish_{timestamp}"
os.makedirs(drive_target_dir, exist_ok=True)

# 3. Model Ağırlıklarını Kopyalayın
!cp -r outputs/slam_asr_cv_turkish_adaptive/best_checkpoint "{drive_target_dir}/"
!cp -r outputs/slam_asr_cv_turkish_adaptive/final_checkpoint "{drive_target_dir}/"

# 4. (Varsa) Değerlendirme Çıktısını Kopyalayın
if os.path.exists("eval_results.jsonl"):
    !cp eval_results.jsonl "{drive_target_dir}/"

# 5. Veri Manifestolarını Kopyalayın (train/val/test jsonl)
if os.path.exists("data/cv"):
    os.makedirs(f"{drive_target_dir}/manifests", exist_ok=True)
    !cp data/cv/*.jsonl "{drive_target_dir}/manifests/"

# 6. Kullanılan Config Dosyasını Kopyalayın
!cp config/hardware_adaptive_turkish.yaml "{drive_target_dir}/"

print("\n" + "=" * 65)
print(f"[*] Tam Yedekleme Başarıyla Tamamlandı!")
print(f"[*] Hedef Dizin: {drive_target_dir}")
print("    • best_checkpoint & final_checkpoint")
print("    • eval_results.jsonl (Test tahminleri ve WER analizi)")
print("    • train/val/test manifestoları")
print("    • hardware_adaptive_turkish.yaml konfigürasyonu")
print("=" * 65)


### 10. Çapraz Korpus Testi: ISSAI Turkish Speech Corpus (ISSAI_TSC) ile Değerlendirme (Out-of-Domain Generalization)
Common Voice Türkçe verisiyle eğitilen modelin, eğitimde hiç görmediği **ISSAI Turkish Speech Corpus (ISSAI_TSC)** test kümesi üzerindeki alan dışı (out-of-domain) genelleme başarımı test edilir.

> **Genelleme Değerlendirmesi:** Model farklı bir akustik ortamda (stüdyo ortamı, kontrollü mikrofon kaydı) test edilerek akustik projektörün genelleme yeteneği ölçülür.

In [ ]:
# 10. Adım: Common Voice ile Eğitilen Modelin ISSAI_TSC Test Seti Üzerinde Değerlendirilmesi
import os
import glob
import json

# 1. Drive'ı Bağlayın (Henüz bağlı değilse)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# 2. Model checkpoint'ini belirleyin (Önce yerel, sonra Drive)
checkpoint_path = "outputs/slam_asr_cv_turkish_adaptive/best_checkpoint"
if not os.path.exists(checkpoint_path):
    candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/slam_asr_tr_cv_*/best_checkpoint"))
    if candidates:
        checkpoint_path = candidates[-1]

# 3. ISSAI_TSC test manifestosunu bulun ve ses dosyalarının diskte var olduğunu doğrulayın
def is_valid_manifest(path):
    if not path or not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                audio_path = data.get("audio_filepath", "")
                return os.path.exists(audio_path)
    except Exception:
        return False
    return False

tsc_manifest = None
manifest_candidates = [
    "data/issai_tsc/test.jsonl",
    "../tsc/data/issai_tsc/test.jsonl",
    "/content/slam-asr-reimplement/tr/tsc/data/issai_tsc/test.jsonl",
]
manifest_candidates.extend(sorted(glob.glob("/content/drive/MyDrive/checkpoints/slam_asr_tr_tsc_*/manifests/test.jsonl")))

for p in manifest_candidates:
    if is_valid_manifest(p):
        tsc_manifest = p
        break

# 4. ISSAI test ses dosyaları diskte yoksa otomatik indir/hazırla
if not tsc_manifest:
    prep_script = "scripts/prepare_issai_tsc.py" if os.path.exists("scripts/prepare_issai_tsc.py") else "../tsc/scripts/prepare_issai_tsc.py"
    if os.path.exists(prep_script):
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
        hf_flag = f'--hf_token "{hf_token}"' if hf_token else ""
        print("[*] ISSAI_TSC test verisi yerel diskte bulunamadı. Hugging Face Hub üzerinden hazırlanıyor...")
        !python "{prep_script}" --download {hf_flag} --output_dir data/issai_tsc
        if is_valid_manifest("data/issai_tsc/test.jsonl"):
            tsc_manifest = "data/issai_tsc/test.jsonl"

# 5. Değerlendirmeyi çalıştır ve sonucu Drive'a kaydet
if tsc_manifest and os.path.exists(checkpoint_path):
    print(f"[*] Kullanılan Checkpoint: {checkpoint_path}")
    print(f"[*] ISSAI_TSC Test Manifestosu: {tsc_manifest}\n")
    !python evaluate.py \
        --config config/hardware_adaptive_turkish.yaml \
        --checkpoint "{checkpoint_path}" \
        --manifest "{tsc_manifest}" \
        --output_file eval_results_tsc_generalization.jsonl \
        --language tr \
        --beam_size 4

    # Değerlendirme çıktısını Drive'daki checkpoint klasörüne yedekle
    drive_target = None
    if "/content/drive/MyDrive" in checkpoint_path:
        drive_target = os.path.dirname(checkpoint_path.rstrip("/"))
    elif "drive_target_dir" in globals() and os.path.exists(drive_target_dir):
        drive_target = drive_target_dir

    if drive_target and os.path.exists(drive_target) and os.path.exists("eval_results_tsc_generalization.jsonl"):
        !cp eval_results_tsc_generalization.jsonl "{drive_target}/"
        print(f"\n[*] ISSAI_TSC değerlendirme sonuçları Drive'a yedeklendi: {drive_target}/eval_results_tsc_generalization.jsonl")
else:
    if not os.path.exists(checkpoint_path):
        print(f"[!] Model checkpoint'i ({checkpoint_path}) bulunamadı. Lütfen önce 6. adımı çalıştırıp modeli eğitin veya Drive'daki bir checkpoint'i kontrol edin.")
    if not tsc_manifest:
        print("[!] ISSAI_TSC test ses dosyaları yerel diskte bulunamadı.")
        print("    (Drive'da test.jsonl manifestosu bulunsa dahi, içindeki ses dosyaları mevcut Colab diskinde yok.)")
        print("    İpucu: Colab Secrets'a (🔑) 'HF_TOKEN' ekleyerek ISSAI verisini otomatik indirebilir veya")
        print("    'data/issai_tsc/test.jsonl' yolunu elle belirtebilirsiniz.")


## 11. Modeli Belleğe Yükle ve Canlı Mikrofon Kaydı ile Konuşmayı Metne Dönüştür
Model GPU belleğine bir kere yüklenir ve hazır bekler. Ardından tarayıcınızdan mikrofona Türkçe konuşursunuz ve model konuşmanızı 1 saniye içinde yazıya döker.

In [ ]:
# 11. Adım: Modeli Belleğe Yükle ve Mikrofon ile Canlı Konuşmayı Metne Dönüştür
import os
import glob
import subprocess
import yaml
from base64 import b64decode
import torch
import IPython.display as ipd
from google.colab import output

from slam_asr.models.slam_model import SLAM_ASR, SLAMASRConfig
from slam_asr.data.audio_processor import AudioProcessor
from slam_asr.utils.device_utils import resolve_auto_hyperparameters

# 1. Drive'ı Bağla ve En Güncel Checkpoint'i Bul
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

checkpoint_candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/slam_asr_cv_turkish_*/best_checkpoint/projector.pt"))
if not checkpoint_candidates:
    checkpoint_candidates = sorted(glob.glob("/content/drive/MyDrive/checkpoints/slam_asr_turkish_*/best_checkpoint/projector.pt"))

if checkpoint_candidates:
    best_ckpt = checkpoint_candidates[-1]
elif os.path.exists("outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt"):
    best_ckpt = "outputs/slam_asr_cv_turkish_adaptive/best_checkpoint/projector.pt"
else:
    raise FileNotFoundError("Hiçbir best_checkpoint/projector.pt dosyası bulunamadı!")

# 2. Modeli GPU'ya Yükle (Zaten Yüklüyse Tekrar Yüklemez!)
if "loaded_slam_model" not in globals():
    print(f"[*] Model checkpoint'i kullanılıyor:\n    {best_ckpt}\n")
    print("[*] 1/3: Model GPU VRAM'ine yükleniyor... Lütfen bekleyin...")
    
    cfg = yaml.safe_load(open("config/hardware_adaptive_turkish.yaml"))
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    model_cfg, data_cfg, _ = resolve_auto_hyperparameters(model_cfg, data_cfg, {})
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_config = SLAMASRConfig(**model_cfg)
    
    model = SLAM_ASR(config=model_config)
    if not getattr(model_config, "load_in_4bit", False) and not getattr(model_config, "load_in_8bit", False):
        model = model.to(device)
    else:
        model.speech_encoder = model.speech_encoder.to(device)
        model.projector = model.projector.to(device)
        
    state_dict = torch.load(best_ckpt, map_location=device)
    model.projector.load_state_dict(state_dict)
    model.eval()
    
    audio_proc = AudioProcessor(
        sampling_rate=data_cfg.get("sampling_rate", 16000),
        is_whisper=data_cfg.get("is_whisper", True),
        whisper_mel_bins=data_cfg.get("whisper_mel_bins", 128),
    )
    
    loaded_slam_model = model
    loaded_audio_proc = audio_proc
    loaded_device = device
    loaded_model_config = model_config
    print("[*] 2/3: Model başarıyla VRAM'e yüklendi ve hazır bekliyor! 🚀\n")
else:
    print("[*] Model zaten VRAM'de hazır bekliyor (Yeniden yükleme yapılmadı). 🚀\n")

# 3. Model Hazır Olduktan Sonra Mikrofon Butonunu Göster
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.target.result);
  reader.readAsDataURL(blob);
});
var recordAudio = () => new Promise(async (resolve, reject) => {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);

    const container = document.createElement("div");
    container.style.cssText = "padding: 12px; margin: 10px 0; background: #1e1e2e; border-radius: 8px; color: white; font-family: sans-serif; display: inline-block;";

    const btn = document.createElement("button");
    btn.innerHTML = "🔴 Kaydı Başlat (Konuşmaya Başlayın)";
    btn.style.cssText = "padding: 12px 24px; font-size: 16px; font-weight: bold; background: #e74c3c; color: white; border: none; border-radius: 6px; cursor: pointer; transition: 0.3s; box-shadow: 0 4px 6px rgba(0,0,0,0.3);";
    container.appendChild(btn);
    document.body.appendChild(container);

    let isRecording = false;

    btn.onclick = () => {
      if (!isRecording) {
        recorder.start();
        isRecording = true;
        btn.innerHTML = "⏹️ Kaydı Bitir (Sonucu Gör)";
        btn.style.background = "#27ae60";
      } else {
        recorder.stop();
        btn.innerHTML = "⏳ Model transkribe ediyor...";
        btn.disabled = true;
        btn.style.background = "#7f8c8d";
      }
    };

    recorder.onstop = async () => {
      const blob = new Blob(chunks, { type: "audio/webm" });
      const base64Text = await b2text(blob);
      container.remove();
      resolve(base64Text);
    };
  } catch (err) {
    reject(err);
  }
});
recordAudio();
"""

print("[*] 3/3: Mikrofon hazır. Konuşmak için aşağıdaki butona basın:")
audio_data = output.eval_js(RECORD_JS)

# 4. Sesi 16 kHz Mono WAV Formatına Dönüştür
webm_file = "live_temp.webm"
wav_file = "live_mic_recording.wav"

with open(webm_file, "wb") as f:
    f.write(b64decode(audio_data.split(",")[1]))

subprocess.run([
    "ffmpeg", "-y", "-i", webm_file,
    "-ar", "16000", "-ac", "1",
    wav_file
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("\n[*] Kaydettiğiniz Ses:")
display(ipd.Audio(wav_file))

# 5. Bellekteki Model ile Anında Çıkarım
speech_tensor = loaded_audio_proc.process(wav_file).unsqueeze(0).to(loaded_device)
use_amp = (loaded_model_config.torch_dtype in ("float16", "bfloat16")) and loaded_device.type == "cuda"
amp_dtype = torch.bfloat16 if loaded_model_config.torch_dtype == "bfloat16" else torch.float16
autocast_ctx = (
    (lambda: torch.amp.autocast("cuda", dtype=amp_dtype))
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast")
    else (lambda: torch.cuda.amp.autocast(dtype=amp_dtype))
)

with torch.no_grad():
    if use_amp:
        with autocast_ctx():
            transcription = loaded_slam_model.generate(
                speech_inputs=speech_tensor,
                prompts=["Konuşmayı metne dönüştürün."],
                num_beams=4,
            )[0]
    else:
        transcription = loaded_slam_model.generate(
            speech_inputs=speech_tensor,
            prompts=["Konuşmayı metne dönüştürün."],
            num_beams=4,
        )[0]

print("\n" + "=" * 55)
print(f"🗣️  Sizin Konuşmanız: {transcription}")
print("=" * 55)
